In [1]:
# Ignore all other GPUs except this one
!export CUDA_VISIBLE_DEVICES=0

In [2]:
import numpy as np         
import matplotlib.pyplot as plt     
from matplotlib.animation import FuncAnimation          
import torch      
import torch.nn as nn    
import torch.optim as optim 
from torch.optim.lr_scheduler import ReduceLROnPlateau 
from torch.utils.data import TensorDataset, DataLoader        
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split 
import time      
from scipy.ndimage import uniform_filter1d    
import pandas as pd
import pickle 
import os
from IPython.display import HTML

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

Assume we have $\phi$ and $n_e$ measurements.\
Want to reconstruct $\bold{E}$ and $\bold{v_e}$

In [4]:
##Plasma Parmeters in normalized units##
n_e = 1.0  # Normalized electron density
T_e = 1.0  # Normalized electron temperature (k_B T_e / eV)
omega_p = 2.0  # Plasma frequency in natural units
v_th = 1.0  # Thermal velocity in natural units
lambda_D = 1.0  # Debye length in natural units
c=1

#normalized length and time grid#
Lx = 1
nx = 200 #number of spatial poitns
dx = Lx/nx 
factor = 10 #how much time wave has to propagate 
x = np.linspace(0,Lx,nx)
dt = .01 # time steps in plasma periods omeg_p**-1
nt = 200 #number of time steps

#normalized wavenumber
k=2 * np.pi/Lx

#Dispersion relation for langmuir waves#
omega = np.sqrt(omega_p**2 + 3*(k**2)*(v_th**2))
tau = factor / omega
phi0 = 1.0

In [5]:
def analytical_solution(x, t, phi0):
    
    phi = phi0*np.cos(k*x-omega*t)
    
    Ex = -k*phi0*np.sin(k*x-omega*t) #Ex = \nabla(phi)
    
    return phi,Ex

def generate_data(nx, nt, Lx, tau, phi0):

    x = np.linspace(0, Lx, nx)
    t = np.linspace(0, tau, nt)

    dx = x[1] - x[0]
    dt = t[1] - t[0]

    x_arr, t_arr =  np.meshgrid(x,t, indexing='ij')

    phi, Ex = analytical_solution(x_arr,t_arr, phi0)

    return x_arr.flatten(), t_arr.flatten(), \
           phi.flatten(), Ex.flatten(), \
           dx, dt

def sparse_measurements(x, t, phi, Ex, num_samples):

    indices = np.random.choice(x.shape[0], num_samples, replace=False)

    return x[indices], t[indices], phi[indices], Ex[indices]

def collocation_points(nx, nt, Lx, tau):

    x = np.linspace(0, Lx, nx)
    t = np.linspace(0, tau, nt)

    x_coll, t_coll = np.meshgrid(x, t, indexing='ij')

    return x_coll.flatten(), t_coll.flatten()

In [6]:
Nx, Nt = 250, 250

X_flat, T_flat, Ex_flat, Phi_flat, dx, dt = generate_data(Nx, Nt, Lx, tau, phi0)
x_sparse, t_sparse, ex_sparse, phi_sparse = sparse_measurements(X_flat, T_flat, \
                                                                Ex_flat, Phi_flat, \
                                                                num_samples=10000)
x_coll, t_coll = collocation_points(Nx,Nt,Lx,tau)

In [ ]:
class Sin(nn.Module):

    def __init__(self):
        super(Sin, self).__init__()

    def forward(self, x):
        return torch.sin(x)
    
class Tanh(nn.Module): 
    
    def __init__(self): 
        super(Tanh,self).__init__()
    
    def forward(self, x): 
        return torch.tanh(x)
    
class Swish(nn.Module): 
    
    def __init__(self): 
        super(Swish, self).__init__()
    
    def forward(self, x): 
        return (x/(1+torch.exp(x)))
    
class Sigmoid(nn.Module): 
    
    def __inint__(self): 
        super(Sigmoid, self).__init__init()
    def forward(self, x): 
        return (1/(1+torch.exp(x)))


class PINN(nn.Module):
    
    def __init__(self, Lx, tau): 
        
        super(PINN, self).__init__()
        
        self.net = nn.Sequential(
            nn.Linear(2,100),
            Tanh(), 
            nn.Linear(100,100),
            Tanh(),
            nn.Linear(100,100),
            Tanh(),
            nn.Linear(100,2)
        )

        self.Lx = Lx
        self.tau = tau
        
    def forward(self, x,  t):
        
        # This operation normalizes the input coordinates to be contained within the range [-1, 1].
        
        x = 2.0*(x / self.Lx) - 1
        t = 2.0*(t / self.tau) - 1
        
        inputs = torch.cat([x,t], dim = 1)
        
        return self.net(inputs)

We are assuming an electrostatic oscillation\
Equation Constraints:\
(1) $mn_e(\frac{\partial\bold{v_e}}{\partial t} + (\bold{v_e}\cdot\nabla)\bold{v_e}) + en_e\bold{E} = 0$\
(2) $\frac{\partial n_{e}}{\partial t} + \nabla \cdot (n_e \bold{v_e}) = 0$\
(3) $\nabla \cdot \bold{E} + \frac{en_e}{\epsilon_0}=0$\
(4) $\bold{E} + \nabla\phi = 0$


In [ ]:
def pinn_loss(model, x_sparse, t_sparse, phi_sparse, means, stds, x_col, t_col):
    
    # Data loss measuring loss of Bdots
    
    sparse_preds = model(x_sparse, t_sparse)
    
    phi_sparse_preds = sparse_preds[:,1].reshape(-1,1)
    phi_sparse_preds = (phi_sparse_preds - means)/stds
    phi_loss = torch.mean(torch.square(phi_sparse_preds - phi_sparse))
    loss_data = phi_loss
    
    
    #Physics loss from collocation points predictin E and B 
    col_preds = model(x_col, t_col) * stds + means
    E_pred = col_preds[:,0]
    phi_pred = col_preds[:,1]
    
    E_pred = E_pred.reshape(-1,1)
    phi_pred = phi_pred.reshape(-1,1)
    
    #time derivatives and gradients
    phi_x = torch.autograd.grad(phi_pred, x_col, grad_outputs=torch.ones_like(phi_pred), create_graph=True, retain_graph=True)[0]
    phi_xx = torch.autograd.grad(phi_x, x_col, grad_outputs=torch.ones_like(phi_x), create_graph=True, retain_graph=True)[0]
    
    ##Poisson's equation 
    #assume we can get rho from phi
    epsilon_0 = 1
    rho = phi_xx * epsilon_0
    loss_poisson = torch.mean((phi_xx + rho/epsilon_0)**2)
    
    ##densitiy continuity 
    
    ##relate phi and E E defined completely by gradient of potential
    loss_2 = torch.mean((E_pred + phi_x)**2)
    
    loss_physics = loss_poisson + loss_2

    return loss_data, loss_physics

In [9]:
# Convert data to PyTorch tensors
x_sparse = torch.tensor(x_sparse.flatten(), dtype=torch.float32, requires_grad=True).reshape(-1,1)
t_sparse = torch.tensor(t_sparse.flatten(), dtype=torch.float32, requires_grad=True).reshape(-1,1)
phi_sparse = torch.tensor(phi_sparse.flatten(), dtype=torch.float32, requires_grad=False).reshape(-1,1)

x_coll = torch.tensor(x_coll.flatten(), dtype=torch.float32).reshape(-1,1).requires_grad_()
t_coll = torch.tensor(t_coll.flatten(), dtype=torch.float32).reshape(-1,1).requires_grad_()

means = torch.stack((torch.mean(phi_sparse, 0).detach(), torch.mean(phi_sparse, 0).detach()), dim=1)
stds = torch.stack((torch.std(phi_sparse, 0).detach(), torch.std(phi_sparse, 0).detach()), dim=1)


means[:,0] = 0.0
stds[:,0] = 1.0

means[:,1] = 0.0
stds[:,1] = 1.0

print('Means = ', means)
print('Stds = ', stds)

# Create DataLoaders
batch_size = int(x_coll.size()[0] // 50)

Means =  tensor([[0., 0.]])
Stds =  tensor([[1., 1.]])


In [10]:
start_time = time.time()
model = PINN(Lx, tau)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=100, verbose=True)

num_epochs = 1000
n_batches = 50


hist = {
    'data_loss': [],
    'physics_loss': [],
    'total_loss': [],
    'learning_rate': []
}

for epoch in range(num_epochs):
    
    epoch_sm_loss = 0
    epoch_phys_loss = 0
    epoch_total_loss = 0
    num_batches = 0

    for i in range(n_batches):

        n_coll = x_coll.shape[0]
        i_idxs = np.random.choice(n_coll, size = n_coll, replace = False)

        x_col_batch = x_coll[i_idxs[i::n_batches],:]
        t_col_batch = t_coll[i_idxs[i::n_batches],:]
        
        optimizer.zero_grad()
        
        sm_loss, phys_loss = pinn_loss(model, x_sparse, t_sparse, phi_sparse, means, stds, x_col_batch, t_col_batch)
     
        total_loss = sm_loss + phys_loss
        
        total_loss.backward()
        optimizer.step()

        epoch_sm_loss += sm_loss.item()
        epoch_phys_loss += phys_loss.item()
        epoch_total_loss += total_loss.item()
        
        num_batches += 1

    # Calculate average losses for the epoch
    avg_sm_loss = epoch_sm_loss / num_batches
    avg_phys_loss = epoch_phys_loss / num_batches
    avg_total_loss = epoch_total_loss / num_batches

    # Step the scheduler
    scheduler.step(avg_total_loss)
    
    # Record history
    hist['data_loss'].append(avg_sm_loss)
    hist['physics_loss'].append(avg_phys_loss)
    hist['total_loss'].append(avg_total_loss)
    hist['learning_rate'].append(optimizer.param_groups[0]['lr'])
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Total Loss: {avg_total_loss:.4f}, "
              f"Sample Measurement Loss: {avg_sm_loss:.4f}, Physics Loss: {avg_phys_loss:.4f}, "
              f"LR: {optimizer.param_groups[0]['lr']:.6f}")
end_time = time.time() 
print("--- %s mins ---" % str(round((end_time-start_time)/60,2)))

/home/aleotz/.conda/envs/pinn_recon/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 10/1000, Total Loss: 13.9923, Sample Measurement Loss: 13.8495, Physics Loss: 0.1428, LR: 0.001000
Epoch 20/1000, Total Loss: 13.7328, Sample Measurement Loss: 13.6913, Physics Loss: 0.0416, LR: 0.001000
Epoch 30/1000, Total Loss: 13.7123, Sample Measurement Loss: 13.6840, Physics Loss: 0.0283, LR: 0.001000
Epoch 40/1000, Total Loss: 13.7061, Sample Measurement Loss: 13.6812, Physics Loss: 0.0249, LR: 0.001000
Epoch 50/1000, Total Loss: 13.6917, Sample Measurement Loss: 13.6792, Physics Loss: 0.0125, LR: 0.001000
Epoch 60/1000, Total Loss: 13.6993, Sample Measurement Loss: 13.6781, Physics Loss: 0.0211, LR: 0.001000
Epoch 70/1000, Total Loss: 13.6908, Sample Measurement Loss: 13.6770, Physics Loss: 0.0139, LR: 0.001000
Epoch 80/1000, Total Loss: 13.6910, Sample Measurement Loss: 13.6767, Physics Loss: 0.0142, LR: 0.001000
Epoch 90/1000, Total Loss: 13.6991, Sample Measurement Loss: 13.6759, Physics Loss: 0.0232, LR: 0.001000
Epoch 100/1000, Total Loss: 13.6887, Sample Measurement

KeyboardInterrupt: 

In [ ]:
# #IC: Small perturbation in electrostatic potential 
# phi = np.sin(k*x)
# E = -np.gradient(phi, dx)
# E_new = np.zeros(nx)
# E_time = np.zeros((nt,nx))

# # Time loop
# for t in range(nt):
#     # Update electric field with normalized frequency
#     E_new = E * np.cos(omega * t * dt)
    
#     # Store electric field for animation
#     E_time[t, :] = E_new
    

# # Create an animation of the wave
# fig, ax = plt.subplots()
# line, = ax.plot(x, E_time[0, :])

# def update(frame):
#     line.set_ydata(E_time[frame, :])
#     return line,
# plt.title("1D Langmuir Wave")
# ani = FuncAnimation(fig, update, frames=nt, interval=50, blit=True)
# HTML(ani.to_jshtml())
